# Goalkeeper feature engineering

This is the dedicated GK research branch of `fantasy_premier_league.ipynb`. The root owns shared data construction and benchmark definitions; this notebook owns goalkeeper-specific features, diagnostics, model comparisons, and prospective challengers.

The extracted benchmark represents availability, save volume, bonus potential, team defence, clean-sheet probability, and opponent attack by venue. Results here do not change the root model automatically.

In [1]:
import pandas as pd
from IPython.display import display

from position_research import compare_models

POSITION = "GK"
K = 5
PANEL_PATH = "fpl_feature_panel.pkl"

panel = pd.read_pickle(PANEL_PATH)
position_panel = panel.loc[panel["position"].eq(POSITION)].copy()
print(POSITION, position_panel.shape, "K =", K)
display(position_panel.groupby("season")["GW"].agg(["min", "max", "nunique"]))

GK (7006, 95) K = 5


,min,max,nunique
season,,,
2024-25,1,38,38
2025-26,1,38,38
2026-27,1,11,11


## Extracted GK feature set

This is the current root challenger moved into its own research branch. Future goalkeeper work should test saves-versus-clean-sheet routes separately, opponent shot quality, penalty exposure, and home/away effects here before promotion.

In [2]:
goalkeeper_features = [
    "minutes_ewm", "zero_min_rate_5", "start_rate_5", "minutes_60_rate",
    "minutes_form", "saves_per90_5", "bps_per90_5", "bonus_rate_5",
    "team_xga_venue_5", "opp_xg_venue_5", "venue_defense_matchup",
    "team_clean_sheet_venue_rate_5", "n_fixtures", "n_home", "value",
]

missing = sorted(set(goalkeeper_features) - set(position_panel.columns))
assert not missing, f"Missing root features: {missing}"
print("GK features:", len(goalkeeper_features))
display(pd.Series(goalkeeper_features, name="feature").to_frame())

GK features: 15


,feature
0,minutes_ewm
1,zero_min_rate_5
2,start_rate_5
3,minutes_60_rate
4,minutes_form
5,saves_per90_5
6,bps_per90_5
7,bonus_rate_5
8,team_xga_venue_5
9,opp_xg_venue_5


## Leak-safe walk-forward benchmark

Each evaluated gameweek trains only on earlier rows. The target and predicted shortlist are tie-aware top K, and each training gameweek receives equal total weight. GW1-19 is development; GW20-38 is a historical check.

In [3]:
goalkeeper_summary, goalkeeper_detail = compare_models(
    position_panel,
    goalkeeper_features,
    K,
)
display(goalkeeper_summary)
display(
    goalkeeper_summary.pivot(index="model", columns="block", values="mean_hit_rate")
)

,block,model,mean_hit_rate,correct,predictions,weeks
1,development_gw1_19,logistic_regression,0.400000,38,95,19
2,development_gw1_19,minutes_sort,0.394403,52,132,19
0,development_gw1_19,gradient_boosting,0.336842,32,95,19
5,historical_check_gw20_38,minutes_sort,0.384211,40,105,19
4,historical_check_gw20_38,logistic_regression,0.378947,36,95,19
3,historical_check_gw20_38,gradient_boosting,0.326316,31,95,19


block,development_gw1_19,historical_check_gw20_38
model,,
gradient_boosting,0.336842,0.326316
logistic_regression,0.400000,0.378947
minutes_sort,0.394403,0.384211
